In [ ]:
import os
import pickle
from math import radians, sin, cos, asin, sqrt
from functools import partial
from collections import defaultdict
from multiprocessing import cpu_count, get_context

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx

from shapely.geometry import Point, LineString, box
from shapely.ops import split

# -------------------------------------------------------------------
# 0) FILE PATHS
#   Relative to the repository root, found by walking up from the working
#   directory (analysis/).
# -------------------------------------------------------------------
def find_repo_root():
    """Nearest folder at or above the working directory holding data/ and
    analysis/Analysis.ipynb."""
    d = os.path.abspath(os.getcwd())
    while True:
        if (os.path.isdir(os.path.join(d, "data"))
                and os.path.isfile(os.path.join(d, "analysis", "Analysis.ipynb"))):
            return d + os.sep
        if os.path.dirname(d) == d:
            raise FileNotFoundError(
                f"no repository root above {os.getcwd()}; run this notebook "
                "from inside the manure-trucking folder")
        d = os.path.dirname(d)


ROOT_PATH = find_repo_root()
# The OSM extract and base pickle are read only if the processed pickle is missing.
OSM_FILE = ROOT_PATH + "data/routing-data/california-260318_study_area.osm"
PICKLE_FILE_BASE = ROOT_PATH + "data/routing-data/study_area_network.pkl"
PICKLE_FILE_PROCESSED = ROOT_PATH + "data/routing-data/study_area_network_processed_v13.pkl"
OD_PICKLE_FILE = ROOT_PATH + "outputs/results/routing/od_table_processed_v13.pkl"

MANURE_FILE = ROOT_PATH + "outputs/review/manifests/processed_manure_manifests.csv"
assert os.path.exists(MANURE_FILE), f"no manifest table at {MANURE_FILE}"
print(f"Manifest table: {MANURE_FILE}")
# Output: manifests with destination-point geometry, recalculated haul counts
# and validation columns.
MANURE_ANALYSIS_FILE = ROOT_PATH + "outputs/results/routing/processed_manure_manifests_analysis.geojson"
BOUNDARY_FILE = ROOT_PATH + "data/geospatial-data/CA_Water_Board_Boundary/CentralValleyRegionalWaterBoard_EPSG4326.geojson"

COUNTIES_FILE = ROOT_PATH + "data/geospatial-data/CA_Counties/CA_Counties.shp"

OUT_SHAPEFILE_DIR = ROOT_PATH + "outputs/results/routing"
OUT_SHAPEFILE_NAME = "network_with_hauls_v13.gpkg"
OUT_OD_LINES = "od_lines_with_status.geojson"
OUT_OD_PAIRS = "od_pairs_with_status.geojson"
# One row per route per road segment; summed over routes it equals edge_agg, so
# any subset of routes can be re-aggregated without re-routing.
OUT_ROUTE_EDGES = "route_edge_usage_v13.csv.gz"

M_PER_KM = 1000.0

# -------------------------------------------------------------------
# HAUL-SIZE CLASSES
# Manifest column names and the truck-size classes for the haul split.
# -------------------------------------------------------------------
MASS_PER_HAUL_COL = "Manure Mass per Haul (tons)"
NUM_HAULS_COL     = "Manure Number of Hauls"
TOTAL_TONS_COL    = "Total Manure Amount (tons)"
HAUL_10_COL       = "Number of 10-ton Hauls for Analysis"
HAUL_20_COL       = "Number of 20-ton Hauls for Analysis"
DEST_LAT_COL      = "Destination Latitude (Geocoded)"
DEST_LON_COL      = "Destination Longitude (Geocoded)"
ORIG_LAT_COL      = "Origin Dairy Latitude (Geocoded)"
ORIG_LON_COL      = "Origin Dairy Longitude (Geocoded)"
DEST_ADDR_COL     = "Destination Address Final"

HAUL_LO_MIN, HAUL_LO_MAX = 5.0, 15.0   # [5, 15) tons per load -> 10-ton trucks
HAUL_HI_MIN              = 15.0        # >= 15  tons per load  -> 20-ton trucks
HAUL_LO_CAPACITY         = 10.0
HAUL_HI_CAPACITY         = 20.0

# Manifests with more than 0 and fewer than this many tons get one 10-ton trip.
MIN_SINGLE_HAUL_TONS = 10.0

# Descriptive manifest fields carried into the OD-line and OD-point exports.
# Each output column takes the first listed manifest column present, else null.
OD_PASSTHROUGH_COLUMNS = {
    "origin_name":    ["Origin Dairy Name"],
    "origin_address": ["Origin Dairy Address"],
    "dest_address":   ["Destination Address Final", "Destination Address"],
    "dest_type":      ["Destination Type (Standardized)", "Destination Type"],
    "source_pdf":     ["Source PDF"],
}

# -------------------------------------------------------------------
# 1) GRAPH LOAD
# -------------------------------------------------------------------
def load_graph_base(OSM_FILE, PICKLE_FILE_BASE):
    """Load the base graph from pickle, or build it from the OSM XML and pickle it."""
    if os.path.exists(PICKLE_FILE_BASE):
        print("Loading base graph from pickle...")
        with open(PICKLE_FILE_BASE, "rb") as f:
            G = pickle.load(f)
        print(f" Loaded pickle graph: {len(G.nodes):,} nodes, {len(G.edges):,} edges")
    else:
        print("Pickle not found; loading from OSM XML...")
        G = ox.graph_from_xml(OSM_FILE, simplify=True, retain_all=False)
        print(f" Loaded OSM graph: {len(G.nodes):,} nodes, {len(G.edges):,} edges")
        
        print("Saving pickle...")
        with open(PICKLE_FILE_BASE, "wb") as f:
            pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f" Saved base pickle to {PICKLE_FILE_BASE}")
    return G
    
# -------------------------------------------------------------------
# 2) MISSING EDGE LENGTHS / SPEEDS
# -------------------------------------------------------------------
def fill_missing_edge_lengths_planar(G):
    """Fill missing edge lengths with planar distance (meters; needs a projected CRS)."""
    fixed = 0
    for u, v, k, data in G.edges(keys=True, data=True):
        length = data.get("length", None)
        if length is None or pd.isna(length):
            geom = data.get("geometry", None)
            if geom is not None and hasattr(geom, "length"):
                data["length"] = float(geom.length)
            else:
                x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
                x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]
                data["length"] = float(((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5)
            fixed += 1
    print(f" Filled {fixed:,} missing edge lengths (meters) using planar fallback.")
    return G

def fill_missing_speeds(G, fallback_kph=40.0):
    fixed = 0
    for _, _, _, data in G.edges(keys=True, data=True):
        spd = data.get("speed_kph", None)
        if spd is None or pd.isna(spd):
            data["speed_kph"] = float(fallback_kph)
            fixed += 1
    print(f" Filled {fixed:,} missing speed_kph using fallback={fallback_kph}.")
    return G

def assert_no_null_length_speed(G):
    null_len = 0
    null_spd = 0
    for _, _, _, data in G.edges(keys=True, data=True):
        if data.get("length", None) is None or pd.isna(data.get("length")):
            null_len += 1
        if data.get("speed_kph", None) is None or pd.isna(data.get("speed_kph")):
            null_spd += 1
    if null_len or null_spd:
        raise ValueError(f"Still have nulls: length={null_len:,}, speed_kph={null_spd:,}")
    print(" Verified: no null edge 'length' or 'speed_kph' remain.")

# -------------------------------------------------------------------
# 3) MANURE -> OD TABLE
# -------------------------------------------------------------------
def load_manure_manifests(path):
    """Read the manifest table (CSV or spatial file) with destination-point geometry.

    Rows without a destination geocode get a null geometry.
    """
    print(f"Loading manifests from {os.path.basename(path)}...")
    if str(path).lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        df = pd.DataFrame(gpd.read_file(path))
    df = df.drop(columns=["geometry"], errors="ignore").reset_index(drop=True)

    lon = pd.to_numeric(df[DEST_LON_COL], errors="coerce")
    lat = pd.to_numeric(df[DEST_LAT_COL], errors="coerce")
    pts = gpd.points_from_xy(lon, lat)
    geom = [None if (pd.isna(x) or pd.isna(y)) else p for p, x, y in zip(pts, lon, lat)]

    gdf = gpd.GeoDataFrame(df, geometry=geom, crs="EPSG:4326")

    # RouteID, the join key for all downstream products, is assigned before any
    # rows are dropped.
    if "RouteID" not in gdf.columns:
        gdf["RouteID"] = gdf.index + 1

    print(f"  {len(gdf):,} manifests, {int(gdf.geometry.notna().sum()):,} with a destination geocode")
    return gdf

def add_haul_estimates(df):
    """Rebuild the 10-ton / 20-ton haul counts from reported tonnage.

    Mass per haul in [HAUL_LO_MIN, HAUL_LO_MAX) t is the 10-ton class and
    >= HAUL_HI_MIN t the 20-ton class. Manifests in a class that report a haul
    count keep it; the tonnage split between the classes estimates hauls for
    all other manifests. Manifests under MIN_SINGLE_HAUL_TONS get one 10-ton
    trip. Existing haul columns are dropped and rebuilt.
    """
    print("\n=== RECALCULATING HAUL COUNTS ===")
    dropped = [c for c in (HAUL_10_COL, HAUL_20_COL) if c in df.columns]
    if dropped:
        print(f"  Dropping existing {dropped} and recalculating")
    df = df.drop(columns=dropped)

    rate   = pd.to_numeric(df[MASS_PER_HAUL_COL], errors="coerce")
    hauls  = pd.to_numeric(df[NUM_HAULS_COL], errors="coerce")
    amount = pd.to_numeric(df[TOTAL_TONS_COL], errors="coerce")

    in_lo = rate.between(HAUL_LO_MIN, HAUL_LO_MAX, inclusive="left")
    in_hi = rate >= HAUL_HI_MIN

    mass_lo = amount[in_lo].sum()
    mass_hi = amount[in_hi].sum()
    total   = mass_lo + mass_hi
    p_lo    = mass_lo / total if total > 0 else 0.5
    p_hi    = 1.0 - p_lo

    print(f"  10-ton class [{HAUL_LO_MIN:g}, {HAUL_LO_MAX:g}) t/load: "
          f"{int(in_lo.sum()):,} manifests, {mass_lo:,.1f} tons")
    print(f"  20-ton class >= {HAUL_HI_MIN:g} t/load:      "
          f"{int(in_hi.sum()):,} manifests, {mass_hi:,.1f} tons")
    print(f"  Tonnage split: {p_lo:.4%} by 10-ton truck, {p_hi:.4%} by 20-ton truck")

    est_lo = (amount * p_lo / HAUL_LO_CAPACITY).round()
    est_hi = (amount * p_hi / HAUL_HI_CAPACITY).round()

    # Use the reported haul count wherever the mass per haul identifies the class.
    reported = rate.notna() & hauls.notna()
    n_hauls  = hauls.round()
    use_lo   = reported & in_lo
    use_hi   = reported & in_hi

    h10 = est_lo.where(~use_lo, n_hauls).where(~use_hi, 0.0)
    h20 = est_hi.where(~use_hi, n_hauls).where(~use_lo, 0.0)

    # Manifests with 0 < tons < MIN_SINGLE_HAUL_TONS get a single 10-ton trip.
    tiny = (amount > 0) & (amount < MIN_SINGLE_HAUL_TONS)
    h10 = h10.where(~tiny, 1.0)
    h20 = h20.where(~tiny, 0.0)

    df[HAUL_10_COL] = h10.astype("Int64")
    df[HAUL_20_COL] = h20.astype("Int64")

    n_est = int((~use_lo & ~use_hi & ~tiny & amount.notna()).sum())
    print(f"  Reported counts used: {int(use_lo.sum()):,} as 10-ton, "
          f"{int(use_hi.sum()):,} as 20-ton")
    print(f"  Estimated from tonnage: {n_est:,} manifests")
    print(f"  Under {MIN_SINGLE_HAUL_TONS:g} tons -> one 10-ton trip: {int(tiny.sum()):,} manifests")
    print(f"  No tonnage reported: {int(amount.isna().sum()):,} manifests left null")
    print(f"  Totals: {df[HAUL_10_COL].sum():,} 10-ton hauls, "
          f"{df[HAUL_20_COL].sum():,} 20-ton hauls")
    return df

def load_od_table(MANURE_FILE, MANURE_ANALYSIS_FILE, COUNTIES_FILE):
    manure_gdf = load_manure_manifests(MANURE_FILE)

    # Rebuild the 10-ton/20-ton haul split from tonnage over all manifests,
    # before any rows are dropped.
    manure_gdf = add_haul_estimates(manure_gdf)

    manure = pd.DataFrame(manure_gdf)

    # a) Total manifests
    total_manifests = len(manure)
    print(f"a) # Total Manifests: {total_manifests:,}")
    
    # b) Drop manifests with no destination address
    dest_addr_col = next((c for c in [DEST_ADDR_COL, "Destination Address"]
                          if c in manure.columns), None)
    if dest_addr_col is None:
        no_dest_addr_mask = pd.Series(False, index=manure.index)
    else:
        addr = manure[dest_addr_col]
        no_dest_addr_mask = addr.isna() | addr.astype(str).str.strip().isin(["", "nan", "None"])

    print(f"b) # dropped with no destination address: {no_dest_addr_mask.sum():,}")
    manure = manure[~no_dest_addr_mask].copy()

    # c) Drop manifests with no destination geocode
    failed_geocode_mask = manure["Destination Latitude (Geocoded)"].isnull() | manure["Destination Longitude (Geocoded)"].isnull()
    failed_geocode = failed_geocode_mask.sum()
    print(f"c) # dropped due to failed geocode: {failed_geocode:,}")
    manure = manure[~failed_geocode_mask].copy()
    
    # Drop manifests with no origin geocode
    orig_failed_mask = manure["Origin Dairy Latitude (Geocoded)"].isnull() | manure["Origin Dairy Longitude (Geocoded)"].isnull()
    manure = manure[~orig_failed_mask].copy()

    # Tag origins and destinations with their county
    print("\nMapping counties to origins and destinations...")
    counties = gpd.read_file(COUNTIES_FILE)
    if counties.crs != "EPSG:4326":
        counties = counties.to_crs("EPSG:4326")
        
    # Detect the county name column
    county_col = next((col for col in ['NAME', 'COUNTY', 'NAMELSAD', 'County'] if col in counties.columns), counties.columns[0])

    # Origin county
    orig_gdf = gpd.GeoDataFrame(
        manure, 
        geometry=gpd.points_from_xy(manure["Origin Dairy Longitude (Geocoded)"], manure["Origin Dairy Latitude (Geocoded)"]), 
        crs="EPSG:4326"
    )
    orig_gdf = gpd.sjoin(orig_gdf, counties[[county_col, 'geometry']], how="left", predicate="intersects")
    orig_gdf = orig_gdf.rename(columns={county_col: 'Origin County'}).drop(columns=['index_right', 'geometry'], errors='ignore')

    # Destination county
    dest_gdf = gpd.GeoDataFrame(
        orig_gdf, 
        geometry=gpd.points_from_xy(orig_gdf["Destination Longitude (Geocoded)"], orig_gdf["Destination Latitude (Geocoded)"]), 
        crs="EPSG:4326"
    )
    dest_gdf = gpd.sjoin(dest_gdf, counties[[county_col, 'geometry']], how="left", predicate="intersects")
    dest_gdf = dest_gdf.rename(columns={county_col: 'Destination County'}).drop(columns=['index_right', 'geometry'], errors='ignore')
    
    manure = dest_gdf
    
    # d) Drop manifests whose APN-derived destination is in another county
    if "Destination Address Final Source" in manure.columns:
        diff_county_apn = (manure["Origin County"] != manure["Destination County"]) & \
                          (manure["Destination Address Final Source"] == "Destination Assessor Parcel Number")
    else:
        diff_county_apn = pd.Series(False, index=manure.index)

    print(f"d) # dropped due to cross-county destination APN: {diff_county_apn.sum():,}")

    manure = manure[~diff_county_apn].copy()

    # Haul counts as numbers, missing -> 0
    if "RouteID" not in manure.columns:
        manure["RouteID"] = manure.index + 1

    manure["Hauls_10ton"] = pd.to_numeric(
        manure["Number of 10-ton Hauls for Analysis"], errors="coerce"
    ).fillna(0)

    manure["Hauls_20ton"] = pd.to_numeric(
        manure["Number of 20-ton Hauls for Analysis"], errors="coerce"
    ).fillna(0)

    # Build the OD table
    od = manure[[
        "RouteID",
        "Origin Dairy Latitude (Geocoded)",
        "Origin Dairy Longitude (Geocoded)",
        "Destination Latitude (Geocoded)",
        "Destination Longitude (Geocoded)",
        "Hauls_10ton",
        "Hauls_20ton",
        "Origin County",
        "Destination County"
    ]].rename(columns={
        "Origin Dairy Latitude (Geocoded)": "orig_lat",
        "Origin Dairy Longitude (Geocoded)": "orig_lon",
        "Destination Latitude (Geocoded)": "dest_lat",
        "Destination Longitude (Geocoded)": "dest_lon",
    })

    # Attach the descriptive passthrough fields, aligned on the shared index.
    for out_col, candidates in OD_PASSTHROUGH_COLUMNS.items():
        src_col = next((c for c in candidates if c in manure.columns), None)
        od[out_col] = manure[src_col] if src_col is not None else None
        print(f"  {out_col:<12} <- {src_col if src_col else 'not found in manifests'}")

    od = od[(od["Hauls_10ton"] > 0) | (od["Hauls_20ton"] > 0)].copy()
    od = od.dropna(subset=["orig_lat", "orig_lon", "dest_lat", "dest_lon"]).reset_index(drop=True)

    print(f" OD pairs with trips: {len(od):,}")
    print(f"  Total 10-ton hauls: {od['Hauls_10ton'].sum():,.0f}")
    print(f"  Total 20-ton hauls: {od['Hauls_20ton'].sum():,.0f}")
    return od, manure_gdf

# -------------------------------------------------------------------
# 4) CLIP OD PAIRS AND GRAPH TO THE STUDY BOUNDARY
# -------------------------------------------------------------------
def filter_od_to_boundary(od, boundary_file):
    boundary = gpd.read_file(boundary_file)
    print(f" Loaded boundary from {boundary_file}")
    print(f"  Boundary CRS: {boundary.crs}")

    if boundary.crs != "EPSG:4326":
        boundary = boundary.to_crs("EPSG:4326")
        print("  Reprojected boundary to EPSG:4326")

    orig_points = gpd.GeoDataFrame(
        od,
        geometry=[Point(lon, lat) for lon, lat in zip(od['orig_lon'], od['orig_lat'])],
        crs="EPSG:4326"
    )

    dest_points = gpd.GeoDataFrame(
        od,
        geometry=[Point(lon, lat) for lon, lat in zip(od['dest_lon'], od['dest_lat'])],
        crs="EPSG:4326"
    )

    orig_within = gpd.sjoin(orig_points, boundary, predicate='within', how='inner')
    dest_within = gpd.sjoin(dest_points, boundary, predicate='within', how='inner')

    valid_indices = orig_within.index.intersection(dest_within.index)
    od_filtered = od.loc[valid_indices].copy()

    removed = len(od) - len(od_filtered)
    print(f" Filtered OD pairs:")
    print(f"  Original: {len(od)}")
    print(f"  Origins outside boundary: {len(od) - len(orig_within)}")
    print(f"  Destinations outside boundary: {len(od) - len(dest_within)}")
    print(f"  Both within boundary: {len(od_filtered)}")
    print(f"  Total removed: {removed}")

    return od_filtered

def clip_graph_to_buffered_boundary(G, boundary_file, buffer_km=10):
    print(f"\n=== CLIPPING GRAPH TO BUFFERED BOUNDARY ===")
    print(f"Buffer: {buffer_km} km")
    
    boundary = gpd.read_file(boundary_file)
    print(f"  Loaded boundary from {boundary_file}")
    print(f"  Original boundary CRS: {boundary.crs}")
    
    boundary_utm = boundary.to_crs("EPSG:32610")  # UTM Zone 10N
    print(f"  Reprojected boundary to EPSG:32610 (UTM 10N) for buffering")
    
    buffer_meters = buffer_km * 1000
    boundary_buffered_utm = boundary_utm.buffer(buffer_meters)
    
    area_original_km2 = boundary_utm.area.sum() / 1e6
    area_buffered_km2 = boundary_buffered_utm.area.sum() / 1e6
    
    print(f"  Original boundary area: {area_original_km2:,.2f} sq km")
    print(f"  Buffered boundary area: {area_buffered_km2:,.2f} sq km")
    print(f"  Buffer added: {area_buffered_km2 - area_original_km2:,.2f} sq km")
    
    boundary_buffered = boundary_buffered_utm.to_crs("EPSG:4326")
    boundary_union = boundary_buffered.union_all()
    
    print(f"  Filtering graph nodes...")
    nodes_to_keep = []
    for node, data in G.nodes(data=True):
        point = Point(data['x'], data['y'])
        if boundary_union.contains(point):
            nodes_to_keep.append(node)
    
    if len(nodes_to_keep) == 0:
        raise ValueError(" No nodes found within buffered boundary!")
    
    G_clipped = G.subgraph(nodes_to_keep).copy()
    
    print(f"  Original graph: {len(G.nodes):,} nodes, {len(G.edges):,} edges")
    print(f"  Clipped graph: {len(G_clipped.nodes):,} nodes, {len(G_clipped.edges):,} edges")
    print(f"  Kept: {len(G_clipped.nodes)/len(G.nodes)*100:.1f}% of nodes")
    
    n_components = nx.number_weakly_connected_components(G_clipped)
    print(f"  Network connectivity: {n_components} weakly connected components")
    
    return G_clipped

# -------------------------------------------------------------------
# 5) ROUTING HELPERS
# -------------------------------------------------------------------
def shortest_path_edges_uvk(G, source, target, weight="travel_time"):
    route_nodes = nx.shortest_path(G, source=source, target=target, weight=weight)

    edges_uvk = []
    for u, v in zip(route_nodes[:-1], route_nodes[1:]):
        keydict = G.get_edge_data(u, v)
        if keydict is None:
            raise RuntimeError(f"No edge data for step {u}->{v}")

        best_k = None
        best_w = float("inf")
        for k, data in keydict.items():
            w = data.get(weight, float("inf"))
            if w < best_w:
                best_w = w
                best_k = k

        edges_uvk.append((u, v, best_k))
    return edges_uvk

# -------------------------------------------------------------------
# 6) EXACT SNAP-TO-EDGE + SPLIT
# -------------------------------------------------------------------
def _edge_linestring(G, u, v, k):
    data = G.edges[u, v, k]
    geom = data.get("geometry", None)
    if geom is not None and hasattr(geom, "length"):
        return geom
    u_xy = (G.nodes[u]["x"], G.nodes[u]["y"])
    v_xy = (G.nodes[v]["x"], G.nodes[v]["y"])
    return LineString([u_xy, v_xy])

def _remove_one_edge(G, u, v, k):
    """Remove a specific edge by key from MultiDiGraph."""
    G.remove_edge(u, v, k)

def add_od_nodes_by_splitting(Gp, od):
    from collections import defaultdict
    import numpy as np
    
    crs_proj = Gp.graph["crs"]

    print("Converting OD coordinates to projected CRS...")
    orig_gdf = gpd.GeoDataFrame(
        od, geometry=gpd.points_from_xy(od.orig_lon, od.orig_lat), crs="EPSG:4326"
    ).to_crs(crs_proj)

    dest_gdf = gpd.GeoDataFrame(
        od, geometry=gpd.points_from_xy(od.dest_lon, od.dest_lat), crs="EPSG:4326"
    ).to_crs(crs_proj)

    print(f"Finding nearest edges for {len(od) * 2:,} points (batch operation)...")
    all_x = np.concatenate([orig_gdf.geometry.x.values, dest_gdf.geometry.x.values])
    all_y = np.concatenate([orig_gdf.geometry.y.values, dest_gdf.geometry.y.values])
    
    try:
        nearest_edges_all = ox.distance.nearest_edges(Gp, all_x, all_y)
    except Exception as e:
        print(f" Error in batch nearest_edges: {e}")
        raise
    
    n_od = len(od)
    edge_to_splits = defaultdict(list)
    
    print("Grouping split points by edge...")
    for i in range(len(od)):
        o_edge = nearest_edges_all[i]
        d_edge = nearest_edges_all[n_od + i]
        
        o_id = -1 * (10_000_000 + int(od.iloc[i]["RouteID"]))
        d_id = -1 * (20_000_000 + int(od.iloc[i]["RouteID"]))
        
        edge_to_splits[o_edge].append((orig_gdf.geometry.iloc[i], o_id, i, True))
        edge_to_splits[d_edge].append((dest_gdf.geometry.iloc[i], d_id, i, False))
    
    print(f"  {len(edge_to_splits):,} unique edges need splitting")
    print(f"  Average {len(od)*2/len(edge_to_splits):.1f} points per edge")
    
    node_assignments = {}
    edges_processed = 0
    edges_skipped = 0
    
    for edge_uvk, split_requests in edge_to_splits.items():
        u, v, k = edge_uvk
        
        if Gp.get_edge_data(u, v, k) is None:
            all_edges = Gp.get_edge_data(u, v)
            if all_edges is not None:
                k = list(all_edges.keys())[0]
            else:
                for point, node_id, od_idx, is_orig in split_requests:
                    u_pt = Point(Gp.nodes[u]["x"], Gp.nodes[u]["y"])
                    v_pt = Point(Gp.nodes[v]["x"], Gp.nodes[v]["y"])
                    nearest = u if point.distance(u_pt) < point.distance(v_pt) else v
                    node_assignments[(od_idx, is_orig)] = nearest
                edges_skipped += 1
                continue
        
        try:
            assigned_nodes = split_edge_at_points(Gp, u, v, k, split_requests)
            node_assignments.update(assigned_nodes)
            edges_processed += 1
        except Exception as e:
            print(f"  Warning: Failed to split edge ({u}, {v}, {k}): {e}")
            for point, node_id, od_idx, is_orig in split_requests:
                u_pt = Point(Gp.nodes[u]["x"], Gp.nodes[u]["y"])
                v_pt = Point(Gp.nodes[v]["x"], Gp.nodes[v]["y"])
                nearest = u if point.distance(u_pt) < point.distance(v_pt) else v
                node_assignments[(od_idx, is_orig)] = nearest
            edges_skipped += 1
        
        if (edges_processed + edges_skipped) % 100 == 0:
            print(f"  Processed {edges_processed + edges_skipped:,}/{len(edge_to_splits):,} edges")
    
    print(f" Split {edges_processed:,} edges successfully (skipped {edges_skipped:,})")
    
    orig_nodes = []
    dest_nodes = []
    valid_indices = []
    
    for i in range(len(od)):
        o_key = (i, True)
        d_key = (i, False)
        
        if o_key in node_assignments and d_key in node_assignments:
            orig_nodes.append(node_assignments[o_key])
            dest_nodes.append(node_assignments[d_key])
            valid_indices.append(i)
        else:
            print(f"  Warning: OD pair {i} missing node assignment")
    
    od_filtered = od.iloc[valid_indices].copy()
    od_filtered["orig_node"] = orig_nodes
    od_filtered["dest_node"] = dest_nodes
    
    skipped_pairs = len(od) - len(od_filtered)
    if skipped_pairs > 0:
        print(f"  Skipped {skipped_pairs} OD pairs due to errors")
    
    print(f" Successfully added nodes for {len(od_filtered):,}/{len(od):,} OD pairs")
    return od_filtered

def split_edge_at_points(G, u, v, k, split_requests, tol=0.5):
    node_assignments = {}
    
    geom = _edge_linestring(G, u, v, k)
    edge_data = dict(G.edges[u, v, k])
    
    projections = []
    for point, node_id, od_idx, is_orig in split_requests:
        dist = geom.project(point)
        snapped = geom.interpolate(dist)
        projections.append((dist, snapped, node_id, od_idx, is_orig))
    
    projections.sort(key=lambda x: x[0])
    
    u_pt = Point(G.nodes[u]["x"], G.nodes[u]["y"])
    v_pt = Point(G.nodes[v]["x"], G.nodes[v]["y"])
    
    points_to_split = []
    for dist, snapped, node_id, od_idx, is_orig in projections:
        if snapped.distance(u_pt) <= tol:
            node_assignments[(od_idx, is_orig)] = u
        elif snapped.distance(v_pt) <= tol:
            node_assignments[(od_idx, is_orig)] = v
        else:
            points_to_split.append((dist, snapped, node_id, od_idx, is_orig))
    
    if not points_to_split:
        return node_assignments
    
    _remove_one_edge(G, u, v, k)
    speed_kph = edge_data.get('speed_kph', 40.0)
    
    prev_node = u
    prev_pt = u_pt
    
    for i, (dist, snapped, node_id, od_idx, is_orig) in enumerate(points_to_split):
        G.add_node(node_id, x=float(snapped.x), y=float(snapped.y))
        node_assignments[(od_idx, is_orig)] = node_id
        
        segment_geom = LineString([prev_pt, snapped])
        segment_length = float(segment_geom.length)
        
        G.add_edge(prev_node, node_id, **{
            'geometry': segment_geom,
            'length': segment_length,
            'speed_kph': speed_kph,
            'travel_time': (segment_length / 1000) / speed_kph * 3600 if speed_kph > 0 else 0
        })
        
        prev_node = node_id
        prev_pt = snapped
    
    final_geom = LineString([prev_pt, v_pt])
    final_length = float(final_geom.length)
    
    G.add_edge(prev_node, v, **{
        'geometry': final_geom,
        'length': final_length,
        'speed_kph': speed_kph,
        'travel_time': (final_length / 1000) / speed_kph * 3600 if speed_kph > 0 else 0
    })
    
    rev_edge_data = G.get_edge_data(v, u)
    if rev_edge_data:
        for rev_k in rev_edge_data.keys():
            _remove_one_edge(G, v, u, rev_k)
            
            prev_node_rev = v
            prev_pt_rev = v_pt
            
            for i in range(len(points_to_split) - 1, -1, -1):
                dist, snapped, node_id, od_idx, is_orig = points_to_split[i]
                
                segment_geom_rev = LineString([prev_pt_rev, snapped])
                segment_length_rev = float(segment_geom_rev.length)
                
                G.add_edge(prev_node_rev, node_id, **{
                    'geometry': segment_geom_rev,
                    'length': segment_length_rev,
                    'speed_kph': speed_kph,
                    'travel_time': (segment_length_rev / 1000) / speed_kph * 3600 if speed_kph > 0 else 0
                })
                
                prev_node_rev = node_id
                prev_pt_rev = snapped
            
            final_geom_rev = LineString([prev_pt_rev, u_pt])
            final_length_rev = float(final_geom_rev.length)
            
            G.add_edge(prev_node_rev, u, **{
                'geometry': final_geom_rev,
                'length': final_length_rev,
                'speed_kph': speed_kph,
                'travel_time': (final_length_rev / 1000) / speed_kph * 3600 if speed_kph > 0 else 0
            })
            break
            
    return node_assignments


# -------------------------------------------------------------------
# 7) OD PROCESSING
# -------------------------------------------------------------------
G_GLOBAL = None  # Graph read by the forked routing workers

def process_od_pair(idx, od):
    global G_GLOBAL
    G = G_GLOBAL
    try:
        row = od.iloc[idx]
        orig_node = int(row["orig_node"])
        dest_node = int(row["dest_node"])
        hauls_10 = float(row["Hauls_10ton"])
        hauls_20 = float(row["Hauls_20ton"])

        if orig_node == dest_node:
            return None, "same_node", 0.0

        path_edges = shortest_path_edges_uvk(G, orig_node, dest_node, weight="travel_time")
        
        edge_usage = {}
        route_total_km = 0.0
        
        for u, v, k in path_edges:
            Lm = float(G.edges[u, v, k].get("length", 0.0))
            km = Lm / M_PER_KM
            route_total_km += km
            
            edge_usage[(u, v, k)] = {
                "hauls_10ton": hauls_10,
                "hauls_20ton": hauls_20,
                "km_10": km * hauls_10,
                "km_20": km * hauls_20,
                "km_tot": km * (hauls_10 + hauls_20),
            }
            
        return edge_usage, "success", route_total_km

    except nx.NetworkXNoPath:
        return None, "no_path", 0.0
    except Exception as e:
        return None, f"error_{str(e)}", 0.0

def run_routing_parallel(Gp, od, num_workers=None):
    if num_workers is None:
        num_workers = max([int((max(1, cpu_count() - 2))/4),1])
    print(f"\n=== ROUTING {len(od):,} PAIRS USING {num_workers} CORES ===")
    
    # Set G_GLOBAL before the pool starts so workers inherit the graph
    global G_GLOBAL
    G_GLOBAL = Gp
    
    edge_agg = defaultdict(lambda: {"hauls_10ton": 0.0, "hauls_20ton": 0.0, "km_10": 0.0, "km_20": 0.0, "km_tot": 0.0})
    status_counts = defaultdict(int)
    routing_results = {}
    
    total = len(od)
    successful = 0
    failed = 0

    chunk_size = 5
    print(f"Using chunk size: {chunk_size} for {num_workers} workers.")
    
    indices = list(range(total))
    
    # Fork explicitly: only fork-started workers inherit G_GLOBAL, and fork is
    # not the default start method on macOS or Python 3.14.
    with get_context("fork").Pool(processes=num_workers) as pool:
        for i in range(0, total, chunk_size):
            chunk_indices = indices[i : i + chunk_size]
            
            worker_func = partial(process_od_pair, od=od)
            results = pool.map(worker_func, chunk_indices)

            for idx, (edge_usage, status, route_kms) in zip(chunk_indices, results):
                status_counts[status] += 1
                
                routing_results[idx] = (edge_usage, status, route_kms)

                if edge_usage is None:
                    failed += 1
                    continue
                successful += 1

                for edge, counts in edge_usage.items():
                    for fld in ("hauls_10ton", "hauls_20ton", "km_10", "km_20", "km_tot"):
                        edge_agg[edge][fld] += counts[fld]

            chunk_end = min(i + chunk_size, total)
            print(f"[{chunk_end/total*100:5.1f}%] {chunk_end:,}/{total:,} "
                  f"(success {successful:,}, failed {failed:,})")

    print("Status counts:", dict(status_counts))
    return edge_agg, status_counts, routing_results


def export_route_edge_usage(od, routing_results, edge_agg, out_dir, out_name):
    """Write the per-route, per-segment haul contributions.

    Keyed by RouteID and (u, v, key), so it joins to the OD products and the
    network file. Asserts that summing over routes reproduces edge_agg (to
    floating-point tolerance).
    """
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, out_name)
    route_ids = od["RouteID"].to_numpy()

    rows = []
    for idx, (edge_usage, status, _km) in routing_results.items():
        if status != "success" or not edge_usage:
            continue
        rid = int(route_ids[idx])
        for (u, v, k), c in edge_usage.items():
            rows.append((rid, u, v, k, c["hauls_10ton"], c["hauls_20ton"],
                         c["km_10"], c["km_20"], c["km_tot"]))

    cols = ["RouteID", "u", "v", "key",
            "hauls_10ton", "hauls_20ton", "km_10", "km_20", "km_tot"]
    df = pd.DataFrame(rows, columns=cols)

    # Check that the table sums to edge_agg on every edge.
    agg = df.groupby(["u", "v", "key"])[
        ["hauls_10ton", "hauls_20ton", "km_10", "km_20", "km_tot"]].sum()
    ref = pd.DataFrame.from_dict(edge_agg, orient="index")
    ref.index = pd.MultiIndex.from_tuples(ref.index, names=["u", "v", "key"])
    ref = ref.reindex(agg.index)
    assert len(agg) == len(edge_agg), (
        f"per-route table covers {len(agg):,} edges, edge_agg has {len(edge_agg):,}")
    for col in ("hauls_10ton", "hauls_20ton", "km_10", "km_20", "km_tot"):
        assert np.allclose(agg[col].to_numpy(), ref[col].to_numpy(),
                           rtol=1e-9, atol=1e-6), \
            f"per-route table does not sum to edge_agg for {col}"

    df.to_csv(out_path, index=False, compression="gzip")
    print(f"\n{out_name}")
    print(f"  {len(df):,} route-segment rows over {df['RouteID'].nunique():,} routes "
          f"and {len(agg):,} segments")
    print(f"  sums to edge_agg exactly; {os.path.getsize(out_path)/1e6:.2f} MB")
    return df


def export_network_with_hauls_shapefile(G, edge_agg, out_dir, out_name):
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, out_name)
    print("\n=== BUILDING OUTPUT FILE ===")
    print("Building edges GeoDataFrame...")
    
    edges_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True, fill_edge_geometry=True)
    print(f" Total edges in network: {len(edges_gdf):,}")

    agg_df = pd.DataFrame.from_dict(edge_agg, orient="index")
    agg_df.index = pd.MultiIndex.from_tuples(agg_df.index, names=["u", "v", "key"])

    agg_df["h10"] = agg_df["hauls_10ton"].astype(float)
    agg_df["h20"] = agg_df["hauls_20ton"].astype(float)
    agg_df["km10"] = agg_df["km_10"].astype(float)
    agg_df["km20"] = agg_df["km_20"].astype(float)
    agg_df["kmtot"] = agg_df["km_tot"].astype(float)
    agg_df = agg_df[["h10", "h20", "km10", "km20", "kmtot"]]

    print("Joining haul/km totals to edges...")
    edges_gdf = edges_gdf.join(agg_df, how="inner")
    
    print(f" Edges with trips: {len(edges_gdf):,}")
    print(f" Reduction: {(1 - len(edges_gdf)/len(G.edges))*100:.1f}% of edges removed")
    
    edges_gdf = edges_gdf.reset_index()

    rename_map = {
        "length": "length_m",
        "speed_kph": "speed_kph",
        "travel_time": "travel_time_sec"
    }
    for old, new in rename_map.items():
        if old in edges_gdf.columns:
            edges_gdf = edges_gdf.rename(columns={old: new})

    keep_cols = ["u", "v", "key", "length_m", "speed_kph", "travel_time_sec", 
                 "name", "highway", "h10", "h20", "km10", "km20", "kmtot", "geometry"]
    keep_cols = [c for c in keep_cols if c in edges_gdf.columns]
    
    edges_simple = edges_gdf[keep_cols]
    edges_simple.to_file(out_path, driver="GPKG")
    
    file_size_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f" Full network written: {file_size_mb:.2f} MB")

def export_od_pairs_with_status(od, routing_results, out_dir, out_name="od_pairs_with_status.geojson"):
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, out_name)
    print(f"\n=== EXPORTING OD PAIRS WITH STATUS ===")

    od_export = od.copy()
    statuses = []
    route_kms = []

    for idx in od_export.index:
        if idx in routing_results:
            # (edge_usage, status, route_total_km) as returned by process_od_pair
            edge_usage, status, km = routing_results[idx]
            statuses.append(status)
            if status != "success" or not edge_usage:
                route_kms.append(np.nan)
            else:
                route_kms.append(float(km) if km is not None else np.nan)
        else:
            statuses.append("not_processed")
            route_kms.append(np.nan)

    od_export['routing_status'] = statuses
    od_export['route_km'] = route_kms
    # Total km travelled over this route by every truck assigned to it.
    od_export['total_truck_km'] = (
        od_export['route_km'] * (od_export['Hauls_10ton'] + od_export['Hauls_20ton']))

    origin_points = gpd.GeoDataFrame(
        od_export,
        geometry=[Point(lon, lat) for lon, lat in zip(od_export['orig_lon'], od_export['orig_lat'])],
        crs="EPSG:4326"
    )
    origin_points['point_type'] = 'origin'

    dest_points = gpd.GeoDataFrame(
        od_export,
        geometry=[Point(lon, lat) for lon, lat in zip(od_export['dest_lon'], od_export['dest_lat'])],
        crs="EPSG:4326"
    )
    dest_points['point_type'] = 'destination'

    all_points = pd.concat([origin_points, dest_points], ignore_index=True)

    export_cols = [
        'RouteID', 'point_type', 'routing_status',
        'origin_name', 'origin_address',
        'dest_address', 'dest_type',
        'source_pdf',
        'orig_lat', 'orig_lon', 'dest_lat', 'dest_lon',
        'Hauls_10ton', 'Hauls_20ton',
        'Origin County', 'Destination County',
        'route_km', 'total_truck_km',
        'valid route', 'validation reason',
        'geometry'
    ]
    export_cols = [c for c in export_cols if c in all_points.columns]
    all_points_export = all_points[export_cols]

    print(f" Writing {len(all_points_export):,} points to {out_path}")
    all_points_export.to_file(out_path, driver="GeoJSON")

    status_counts = all_points_export.groupby(['point_type', 'routing_status']).size()
    print("\n Status Summary:")
    print(status_counts)
    print(f" OD pairs exported to {out_path}")
    return out_path

def _sq_dist(a, b):
    return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

def _route_geometry(G, edges_uvk):
    """Stitch the routed edge geometries into one LineString, in travel order.

    Each edge geometry is oriented to run u->v, and the node shared by
    consecutive edges is written once. Returns None if no usable geometry.
    """
    coords = []
    for u, v, k in edges_uvk:
        geom = _edge_linestring(G, u, v, k)
        pts = list(geom.coords)
        if len(pts) < 2:
            continue

        u_xy = (G.nodes[u]["x"], G.nodes[u]["y"])
        v_xy = (G.nodes[v]["x"], G.nodes[v]["y"])
        forward = _sq_dist(pts[0], u_xy) + _sq_dist(pts[-1], v_xy)
        reverse = _sq_dist(pts[0], v_xy) + _sq_dist(pts[-1], u_xy)
        if reverse < forward:
            pts.reverse()

        if coords and _sq_dist(coords[-1], pts[0]) < 1e-6:
            pts = pts[1:]
        coords.extend(pts)

    if len(coords) < 2:
        return None
    return LineString(coords)

def export_od_lines_with_status(G, od, routing_results, out_dir, out_name="od_lines_with_status.geojson"):
    """Export one routed line per OD pair with its status, manifest fields and km.

    Pairs that did not route get a null geometry, so every OD pair is written.
    """
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, out_name)
    print(f"\n=== EXPORTING OD LINES WITH STATUS ===")

    od_export = od.copy()

    statuses = []
    route_kms = []
    geometries = []
    n_missing_geom = 0

    for idx in od_export.index:
        if idx not in routing_results:
            statuses.append("not_processed")
            route_kms.append(np.nan)
            geometries.append(None)
            continue

        # (edge_usage, status, route_total_km) as returned by process_od_pair.
        # edge_usage keys are the routed edges, in travel order.
        edge_usage, status, km = routing_results[idx]
        statuses.append(status)

        if status != "success" or not edge_usage:
            route_kms.append(np.nan)
            geometries.append(None)
            continue

        route_kms.append(float(km) if km is not None else np.nan)
        geom = _route_geometry(G, list(edge_usage.keys()))
        if geom is None:
            n_missing_geom += 1
        geometries.append(geom)

    od_export["routing_status"] = statuses
    od_export["route_km"] = route_kms

    # Total km travelled over this route by every truck assigned to it.
    od_export["total_truck_km"] = (
        od_export["route_km"] * (od_export["Hauls_10ton"] + od_export["Hauls_20ton"]))

    print(f" Built routed geometry for {sum(g is not None for g in geometries):,} pairs")
    if n_missing_geom:
        print(f"  Warning: {n_missing_geom:,} routed pairs produced no usable geometry")

    # Geometries come from the projected graph; reproject to WGS84 for output.
    crs_proj = G.graph.get("crs", None)
    lines_gdf = gpd.GeoDataFrame(od_export, geometry=geometries, crs=crs_proj)
    if crs_proj is not None:
        lines_gdf = lines_gdf.to_crs("EPSG:4326")

    export_cols = [
        "RouteID", "routing_status",
        "origin_name", "origin_address",
        "dest_address", "dest_type",
        "source_pdf",
        "orig_lat", "orig_lon", "dest_lat", "dest_lon",
        "Hauls_10ton", "Hauls_20ton",
        "Origin County", "Destination County",
        "route_km", "total_truck_km",
        "valid route", "validation reason",
        "geometry",
    ]
    missing = [c for c in export_cols if c not in lines_gdf.columns]
    if missing:
        print(f"  Warning: column(s) not available, omitted from output: {missing}")
    lines_export = lines_gdf[[c for c in export_cols if c in lines_gdf.columns]]

    print(f" Writing {len(lines_export):,} OD lines to {out_path}")
    lines_export.to_file(out_path, driver="GeoJSON")

    print("\n Status Summary:")
    print(lines_export["routing_status"].value_counts())

    routed = lines_export[lines_export["routing_status"] == "success"]
    if len(routed) > 0:
        print(f"\n Routed pairs: {len(routed):,}")
        print(f"  Mean route length:      {routed['route_km'].mean():,.1f} km")
        print(f"  Median route length:    {routed['route_km'].median():,.1f} km")
        print(f"  Longest route:          {routed['route_km'].max():,.1f} km")
        print(f"  Total truck km:         {routed['total_truck_km'].sum():,.0f}")

    file_size_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f" OD lines exported to {out_path} ({file_size_mb:.2f} MB)")
    return out_path

# -------------------------------------------------------------------
# 8) OD VALIDATION
# -------------------------------------------------------------------
VALID_COL  = "valid route"
REASON_COL = "validation reason"

REASON_NO_DEST      = "No destination address"
REASON_OUTSIDE_WB   = "Outside water board"
REASON_FAILED_GEO   = "Failed geocode"
REASON_CROSS_COUNTY = "Cross-county APN"
REASON_ROUTING      = "Routing Failed"

# Checks run in this order and each manifest records only its first failure, so
# reasons are mutually exclusive and survivors after each stage follow by
# subtraction.
REASON_ORDER = [REASON_NO_DEST, REASON_OUTSIDE_WB, REASON_FAILED_GEO,
                REASON_CROSS_COUNTY, REASON_ROUTING]

def points_in_polygons(lon, lat, index, polygons):
    """Index labels whose lon/lat point intersects any polygon in `polygons`."""
    pts = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(lon, lat), index=index, crs="EPSG:4326")
    joined = gpd.sjoin(pts, polygons[["geometry"]], how="inner", predicate="intersects")
    return pd.Index(joined.index.unique())

def assign_polygon_value(lon, lat, index, polygons, value_col):
    """Tag each lon/lat point with `value_col` of the polygon it falls in.

    Points in more than one polygon keep the first match, so the row count is
    unchanged.
    """
    pts = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(lon, lat), index=index, crs="EPSG:4326")
    joined = gpd.sjoin(pts, polygons[[value_col, "geometry"]], how="left", predicate="intersects")
    joined = joined[~joined.index.duplicated(keep="first")]
    return joined[value_col]

def add_validation_columns(manure, od, routing_results, boundary_file, counties_file):
    """Record, per manifest, whether its OD pair passed every check.

    Adds 'valid route' (bool), 'validation reason' (null when valid) and
    'routing_status' from the routing run.
    """
    print("\n=== VALIDATING OD PAIRS ===")
    n_total = len(manure)

    orig_lat = pd.to_numeric(manure[ORIG_LAT_COL], errors="coerce")
    orig_lon = pd.to_numeric(manure[ORIG_LON_COL], errors="coerce")
    dest_lat = pd.to_numeric(manure[DEST_LAT_COL], errors="coerce")
    dest_lon = pd.to_numeric(manure[DEST_LON_COL], errors="coerce")
    has_coords = orig_lat.notna() & orig_lon.notna() & dest_lat.notna() & dest_lon.notna()
    coord_idx = manure.index[has_coords]

    # --- check 1: destination address present
    dest_addr_col = next((c for c in [DEST_ADDR_COL, "Destination Address"]
                          if c in manure.columns), None)
    if dest_addr_col is None:
        print("  Warning: no destination address column; skipping check 1.")
        mask_no_dest = pd.Series(False, index=manure.index)
    else:
        addr = manure[dest_addr_col]
        mask_no_dest = addr.isna() | addr.astype(str).str.strip().isin(["", "nan", "None"])
    print(f"  1) no destination address : {int(mask_no_dest.sum()):,}")

    # --- check 2: both endpoints inside the water board
    boundary = gpd.read_file(boundary_file)
    if boundary.crs != "EPSG:4326":
        boundary = boundary.to_crs("EPSG:4326")
    orig_in = points_in_polygons(orig_lon[has_coords], orig_lat[has_coords], coord_idx, boundary)
    dest_in = points_in_polygons(dest_lon[has_coords], dest_lat[has_coords], coord_idx, boundary)
    both_in = orig_in.intersection(dest_in)
    mask_outside_wb = pd.Series(False, index=manure.index)
    mask_outside_wb.loc[coord_idx] = ~coord_idx.isin(both_in)
    print(f"  2) outside water board    : {int(mask_outside_wb.sum()):,}")

    # --- check 3: geocodes present (rows without coords are not flagged by 2)
    mask_failed_geo = ~has_coords
    print(f"  3) missing a geocode      : {int(mask_failed_geo.sum()):,}")

    # --- check 4: cross-county destination APN
    counties = gpd.read_file(counties_file)
    if counties.crs != "EPSG:4326":
        counties = counties.to_crs("EPSG:4326")
    county_col = next((c for c in ["NAME", "COUNTY", "NAMELSAD", "County"]
                       if c in counties.columns), counties.columns[0])
    orig_county = assign_polygon_value(
        orig_lon[has_coords], orig_lat[has_coords], coord_idx, counties, county_col
    ).reindex(manure.index)
    dest_county = assign_polygon_value(
        dest_lon[has_coords], dest_lat[has_coords], coord_idx, counties, county_col
    ).reindex(manure.index)
    if "Destination Address Final Source" in manure.columns:
        from_apn = manure["Destination Address Final Source"].eq("Destination Assessor Parcel Number")
        mask_cross_county = (orig_county != dest_county) & from_apn
    else:
        mask_cross_county = pd.Series(False, index=manure.index)
    print(f"  4) cross-county dest APN  : {int(mask_cross_county.sum()):,}")

    # --- check 5: the pair routed successfully
    status_by_route = {}
    route_ids = od["RouteID"].tolist() if od is not None and "RouteID" in od.columns else []
    for idx, result in (routing_results or {}).items():
        if 0 <= idx < len(route_ids):
            status_by_route[int(route_ids[idx])] = result[1]

    routing_status = manure["RouteID"].map(
        lambda r: status_by_route.get(int(r)) if pd.notna(r) else None)

    # Manifests that cleared checks 1-4 but have no routing record are labelled
    # snapping_failed.
    cleared = ~(mask_no_dest | mask_outside_wb | mask_failed_geo | mask_cross_county)
    routing_status = routing_status.mask(cleared & routing_status.isna(), "snapping_failed")
    mask_routing_failed = routing_status.ne("success")
    print(f"  5) failed to route        : {int((mask_routing_failed & cleared).sum()):,}")

    # --- resolve the first failing check
    masks = {
        REASON_NO_DEST:      mask_no_dest,
        REASON_OUTSIDE_WB:   mask_outside_wb,
        REASON_FAILED_GEO:   mask_failed_geo,
        REASON_CROSS_COUNTY: mask_cross_county,
        REASON_ROUTING:      mask_routing_failed,
    }
    reason = pd.Series("", index=manure.index, dtype=object)
    for r in REASON_ORDER:
        reason.loc[masks[r].fillna(False).astype(bool) & reason.eq("")] = r

    manure[VALID_COL] = reason.eq("")
    manure[REASON_COL] = reason.where(~reason.eq(""), None)
    manure["routing_status"] = routing_status

    n_valid = int(manure[VALID_COL].sum())
    print("\n" + "=" * 66)
    print("MANURE OD VALIDATION SUMMARY")
    print("=" * 66)
    print(f"{'Total manifests':<34}{n_total:>10,}")
    remaining = n_total
    for r in REASON_ORDER:
        dropped = int(reason.eq(r).sum())
        remaining -= dropped
        print(f"  - dropped: {r:<28}{dropped:>8,}   remaining {remaining:>8,}")
    print("-" * 66)
    print(f"{'valid route = True':<34}{n_valid:>10,}  ({n_valid / n_total * 100:.1f}%)")
    print(f"{'valid route = False':<34}{n_total - n_valid:>10,}")
    print("=" * 66)
    assert remaining == n_valid, "Sequential drop counts do not reconcile with 'valid route'."
    assert len(manure) == n_total, "Row count changed during validation."
    return manure

OD_VALIDATION_COLS = [VALID_COL, REASON_COL]

def attach_validation_to_od(od, manure):
    """Copy the validation columns onto the OD table by RouteID."""
    lookup = manure.set_index("RouteID")
    rid = pd.to_numeric(od["RouteID"], errors="coerce")
    for col in OD_VALIDATION_COLS:
        if col in lookup.columns:
            od[col] = rid.map(lookup[col])
    return od

# -------------------------------------------------------------------
# MAIN EXECUTION
# -------------------------------------------------------------------
if __name__ == "__main__":
    print("=" * 60)
    
    # -------------------------------------------------------------------
    # 1) NETWORK GRAPH (load the processed pickle, or build it)
    # -------------------------------------------------------------------
    if os.path.exists(PICKLE_FILE_PROCESSED):
        print("Loading PROCESSED network graph from pickle...")
        with open(PICKLE_FILE_PROCESSED, "rb") as f:
            Gp = pickle.load(f)
        print(f" Loaded processed graph: {len(Gp.nodes):,} nodes, {len(Gp.edges):,} edges")
    else:
        print("Processed network pickle not found. Building from scratch...")
        
        G = load_graph_base(OSM_FILE, PICKLE_FILE_BASE)

        print("Clip graph to buffered study boundary")

        # Clip to the study boundary buffered by 25 km
        G = clip_graph_to_buffered_boundary(G, BOUNDARY_FILE, buffer_km=25)
        
        n_components = nx.number_weakly_connected_components(G)
        print(f" Network connectivity: {n_components} weakly connected components")
        if n_components > 100:
            print(f" Warning: Network has {n_components} disconnected components")
            print(f" Consider keeping only largest component if routing fails often.")

        print("Projecting graph to UTM...")
        Gp = ox.project_graph(G)
        print(f" Projected CRS: {Gp.graph['crs']}")

        speed_defaults = {
            "motorway": 100, "motorway_link": 70, "trunk": 90, "trunk_link": 60,
            "primary": 80, "primary_link": 50, "secondary": 60, "secondary_link": 40,
            "tertiary": 50, "tertiary_link": 30, "unclassified": 40, "residential": 30,
            "living_street": 20, "service": 20, "pedestrian": 5, "track": 20, "road": 40,
        }
        print("Adding edge speeds...")
        Gp = ox.add_edge_speeds(Gp, hwy_speeds=speed_defaults, fallback=40)

        Gp = fill_missing_edge_lengths_planar(Gp)
        Gp = fill_missing_speeds(Gp, fallback_kph=40)
        assert_no_null_length_speed(Gp)

        print("Adding travel times...")
        Gp = ox.add_edge_travel_times(Gp)
        print(" Network ready (projected, with travel_time).")

        print("\n=== SAVING PROCESSED NETWORK ===")
        with open(PICKLE_FILE_PROCESSED, "wb") as f:
            pickle.dump(Gp, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f" Saved processed graph to {PICKLE_FILE_PROCESSED}")

    # -------------------------------------------------------------------
    # 2) OD TABLE (rebuilt on every run)
    # -------------------------------------------------------------------
    print("\n=== PROCESSING OD TABLE (Fresh Run) ===")
    od, manure_gdf = load_od_table(MANURE_FILE, MANURE_ANALYSIS_FILE, COUNTIES_FILE)

    print("\n=== FILTERING TO BOUNDARY ===")
    od = filter_od_to_boundary(od, BOUNDARY_FILE)
    if len(od) == 0:
        raise ValueError(" No OD pairs remain after boundary filtering! Check your boundary file.")

    print("\n=== ADDING EXACT OD NODES BY EDGE-SPLITTING (BIDIR) ===")
    # Insert each origin and destination into Gp as a node on its nearest edge
    od = add_od_nodes_by_splitting(Gp, od)

    # Positional 0..N-1 index, matching the keys of routing_results
    od = od.reset_index(drop=True)

    # Save the OD table for reference
    with open(OD_PICKLE_FILE, "wb") as f:
        pickle.dump(od, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f" Saved latest OD table to {OD_PICKLE_FILE} for reference.")
    
    print(f"\n=== GRAPH CONNECTIVITY ===")
    print(f"Is strongly connected: {nx.is_strongly_connected(Gp)}")

    # -------------------------------------------------------------------
    # 3) ROUTING
    # -------------------------------------------------------------------
    # Least-travel-time route for every OD pair, in parallel
    edge_agg, status_counts, routing_results = run_routing_parallel(Gp, od)

    # -------------------------------------------------------------------
    # 4) VALIDATION + SPATIAL MANIFEST LAYER
    # -------------------------------------------------------------------
    manure_gdf = add_validation_columns(
        manure_gdf, od, routing_results, BOUNDARY_FILE, COUNTIES_FILE)
    manure_gdf.to_file(MANURE_ANALYSIS_FILE, driver="GeoJSON")
    print(f"\n Wrote validated spatial manifests to {MANURE_ANALYSIS_FILE}")

    # Carry the validation columns onto the OD table so both OD exports report
    # them per route.
    od = attach_validation_to_od(od, manure_gdf)

    # -------------------------------------------------------------------
    # 5) EXPORTS
    # -------------------------------------------------------------------
    print("\n" + "=" * 60)
    export_network_with_hauls_shapefile(Gp, edge_agg, OUT_SHAPEFILE_DIR, OUT_SHAPEFILE_NAME)
    export_route_edge_usage(od, routing_results, edge_agg,
                            OUT_SHAPEFILE_DIR, OUT_ROUTE_EDGES)
    export_od_pairs_with_status(od, routing_results, OUT_SHAPEFILE_DIR, OUT_OD_PAIRS)
    export_od_lines_with_status(Gp, od, routing_results, OUT_SHAPEFILE_DIR, OUT_OD_LINES)

    print("\n" + "=" * 60)
    print(" ANALYSIS COMPLETE!")
    print("=" * 60)

    
